# HAVEN — Data Retrieval Exploration
**Validating data sources before structuring the project**

Data sources used:
- **OpenWeatherMap One Call API 3.0** — current conditions + national alerts (AEMET for Spain)
- **OpenWeatherMap Forecast API 5 days/3h** — extended forecast (free tier)
- **SQLite** — local database schema validation

> One Call 3.0 requires the 'One Call by Call' subscription (free up to 1,000 calls/day).  
> Set your daily limit at: https://home.openweathermap.org/subscriptions

## 0. Setup

In [467]:
%pip install requests python-dotenv pandas --quiet

Note: you may need to restart the kernel to use updated packages.


In [1]:
import requests
import json
import sqlite3
import pandas as pd
from datetime import datetime
from pprint import pprint
import pytest

print('Imports OK')

Imports OK


## 1. Configuration

Register your free API key at: https://openweathermap.org/api  
Activate One Call 3.0 at: https://home.openweathermap.org/subscriptions  

Load from a `.env` file

In [ ]:
# Create a .env file with: OWM_API_KEY=your_key
from dotenv import load_dotenv
import os
load_dotenv()
OWM_API_KEY = os.getenv('OWM_API_KEY')

# Location
CITY    = 'Zaragoza'
COUNTRY = 'Spain'
LAT     = 41.64937507947522
LON     = -0.8937800462407269
UNITS   = 'metric'

print(f'Config loaded | City: {CITY} | Country: {COUNTRY} | lat={LAT}, lon={LON}')

---
## 2. One Call API 3.0 — Current Conditions + National Alerts

Single endpoint returning current weather **and** active national alerts from AEMET (Spain).  
Docs: https://openweathermap.org/api/one-call-3

In [3]:
def get_onecall(api_key, lat, lon, units='metric'):
    """
    One Call API 3.0 — returns current conditions and national alerts.
    minutely/hourly/daily excluded to minimise payload.
    """
    url = 'https://api.openweathermap.org/data/3.0/onecall'
    params = {
        'lat':     lat,
        'lon':     lon,
        'appid':   api_key,
        'units':   units,
        'exclude': 'minutely,hourly,daily',
    }
    response = requests.get(url, params=params, timeout=10)
    response.raise_for_status()
    return response.json()


raw_onecall = get_onecall(OWM_API_KEY, LAT, LON)

print('One Call API 3.0 response received')
print(f'Keys in response: {list(raw_onecall.keys())}')
print(f'Active alerts:    {len(raw_onecall.get("alerts", []))}')

One Call API 3.0 response received
Keys in response: ['lat', 'lon', 'timezone', 'timezone_offset', 'current', 'alerts']
Active alerts:    1


In [ ]:
from datetime import timezone

def parse_current(raw):
    """Extract relevant fields from One Call current block."""
    c = raw.get('current', {})
    return {
        'timestamp':           datetime.fromtimestamp(c['dt'], tz=timezone.utc).strftime('%Y-%m-%dT%H:%M:%S'),
        'city':                raw.get('timezone', '').split('/')[-1],
        'temp_c':              c.get('temp'),
        'feels_like_c':        c.get('feels_like'),
        'humidity_pct':        c.get('humidity'),
        'pressure_hpa':        c.get('pressure'),
        'visibility_m':        c.get('visibility'),
        'wind_speed_ms':       c.get('wind_speed'),
        'wind_gust_ms':        c.get('wind_gust'),
        'rain_1h_mm':          c.get('rain', {}).get('1h', 0),
        'snow_1h_mm':          c.get('snow', {}).get('1h', 0),
        'weather_id':          c.get('weather', [{}])[0].get('id'),
        'weather_main':        c.get('weather', [{}])[0].get('main'),
        'weather_description': c.get('weather', [{}])[0].get('description'),
        'clouds_pct':          c.get('clouds'),
        'uvi':                 c.get('uvi'),
    }


weather = parse_current(raw_onecall)

print('Parsed current conditions:\n')
for k, v in weather.items():
    print(f'  {k:<28} {v}')

In [6]:
# View as DataFrame
pd.DataFrame([weather]).T.rename(columns={0: 'value'})

,value
timestamp,2026-04-21T07:21:48
city,Madrid
temp_c,17.63
feels_like_c,16.75
humidity_pct,50
pressure_hpa,1011
visibility_m,10000
wind_speed_ms,1.03
wind_gust_ms,None
rain_1h_mm,0


---
## 3. One Call API 3.0 — National Alerts

Alerts are issued by **AEMET** for Spain.  

Note: One Call 3.0 does not expose severity as a label (Yellow/Orange/Red) directly —  
we infer it from the `tags` field and feed it into the risk engine.

In [ ]:
def infer_severity_from_tags(tags):
    """
    Infer severity label from One Call alert tags.
    Returns: Minor / Moderate / Severe / Extreme / Unknown
    """
    tags_lower = [t.lower() for t in (tags or [])]
    if any(t in tags_lower for t in ['extreme', 'tornado', 'hurricane', 'tsunami']):
        return 'Extreme'
    elif any(t in tags_lower for t in ['thunderstorm', 'snow', 'flood', 'wind', 'rain', 'fog']):
        return 'Moderate'
    elif tags:
        return 'Minor'
    return 'Unknown'


def parse_alerts(raw):
    """Extract and normalise alerts from One Call 3.0 response."""
    parsed = []
    for a in raw.get('alerts', []):
        tags     = a.get('tags', [])
        severity = infer_severity_from_tags(tags)
        parsed.append({
            'title':       a.get('event', ''),
            'event':       a.get('event', ''),
            'sender_name': a.get('sender_name', ''),
            'severity':    severity,
            'urgency':     'Expected',
            'certainty':   'Likely',
            'onset':       datetime.fromtimestamp(a['start'], tz=timezone.utc).strftime('%Y-%m-%dT%H:%M:%S') if 'start' in a else '',
            'expires':     datetime.fromtimestamp(a['end'],   tz=timezone.utc).strftime('%Y-%m-%dT%H:%M:%S') if 'end'   in a else '',
            'description': a.get('description', ''),
            'tags':        tags,
        })
    return parsed


alerts = parse_alerts(raw_onecall)

print(f'Active alerts: {len(alerts)}')
if alerts:
    for a in alerts:
        print(f"\n  [{a['sender_name']}] {a['event']}")
        print(f"  Severity : {a['severity']}")
        print(f"  Tags     : {a['tags']}")
        print(f"  Onset    : {a['onset']}")
        print(f"  Expires  : {a['expires']}")
        print(f"  Desc     : {a['description'][:160]}")
else:
    print('  No active alerts — normal under calm conditions.')

In [8]:
# Show alerts as DataFrame
if alerts:
    display(pd.DataFrame(alerts).drop(columns=['description']))
else:
    cols = ['title', 'event', 'sender_name', 'severity', 'urgency', 'certainty', 'onset', 'expires', 'tags']
    print('No active alerts — empty DataFrame schema:')
    print(pd.DataFrame(columns=cols))

,title,event,sender_name,severity,urgency,certainty,onset,expires,tags
0,Moderate thunderstorm warning,Moderate thunderstorm warning,AEMET. Agencia Estatal de Meteorología,Moderate,Expected,Likely,2026-04-21T14:00:00,2026-04-21T19:59:59,[Thunderstorm]


---
## 4. Forecast API — 5 Days / 3-Hour Steps

Separate free-tier endpoint — 40 timestamps over 5 days.  
Used for multi-day risk projection in the risk engine.  
Docs: https://openweathermap.org/forecast5

In [9]:
def get_forecast(api_key, lat, lon, units='metric'):
    """5-day / 3-hour forecast — free tier endpoint."""
    url = 'https://api.openweathermap.org/data/2.5/forecast'
    params = {'lat': lat, 'lon': lon, 'appid': api_key, 'units': units}
    response = requests.get(url, params=params, timeout=10)
    response.raise_for_status()
    return response.json()


def parse_forecast(raw):
    """Convert forecast list to clean records."""
    records = []
    for item in raw['list']:
        weather_list = item.get('weather', [])

        # OWM returns a list but almost always with one element.
        # We take the first (most severe by convention) and log if there are more.
        if len(weather_list) > 1:
            print(f"[NOTE] Multiple weather conditions at {item['dt_txt']}: "
                  f"{[w['id'] for w in weather_list]} — using first.")

        primary_weather = weather_list[0] if weather_list else {}

        records.append({
            'timestamp':           item['dt_txt'],
            'temp_c':              item['main']['temp'],
            'temp_min_c':          item['main']['temp_min'],
            'temp_max_c':          item['main']['temp_max'],
            'humidity_pct':        item['main']['humidity'],
            'wind_speed_ms':       item['wind']['speed'],
            'wind_gust_ms':        item['wind'].get('gust'),
            'rain_3h_mm':          item.get('rain', {}).get('3h', 0),
            'snow_3h_mm':          item.get('snow', {}).get('3h', 0),
            'weather_id':          primary_weather.get('id'),
            'weather_main':        primary_weather.get('main'),
            'weather_description': primary_weather.get('description'),
            'weather_all_ids':     [w['id'] for w in weather_list],  # full list for reference
            'clouds_pct':          item['clouds']['all'],
            'pop':                 item.get('pop', 0),
        })
    return records


raw_forecast     = get_forecast(OWM_API_KEY, LAT, LON)
forecast_records = parse_forecast(raw_forecast)
df_forecast      = pd.DataFrame(forecast_records)
df_forecast['timestamp'] = pd.to_datetime(df_forecast['timestamp'])

print(f'Forecast received: {len(df_forecast)} timestamps')
print(f'Period: {df_forecast["timestamp"].iloc[0]} → {df_forecast["timestamp"].iloc[-1]}')
df_forecast.head(8)

Forecast received: 40 timestamps
Period: 2026-04-21 09:00:00 → 2026-04-26 06:00:00


,timestamp,temp_c,temp_min_c,temp_max_c,humidity_pct,wind_speed_ms,wind_gust_ms,rain_3h_mm,snow_3h_mm,weather_id,weather_main,weather_description,weather_all_ids,clouds_pct,pop
0,2026-04-21 09:00:00,18.40,18.40,19.94,52,1.63,1.83,0.00,0,802,Clouds,scattered clouds,[802],27,0.00
1,2026-04-21 12:00:00,22.96,22.96,25.62,41,3.61,3.52,0.00,0,803,Clouds,broken clouds,[803],55,0.00
2,2026-04-21 15:00:00,29.10,29.10,29.10,27,4.90,5.11,0.00,0,803,Clouds,broken clouds,[803],75,0.00
3,2026-04-21 18:00:00,26.00,26.00,26.00,33,5.99,9.85,0.00,0,803,Clouds,broken clouds,[803],60,0.00
4,2026-04-21 21:00:00,21.77,21.77,21.77,44,3.24,3.60,0.11,0,500,Rain,light rain,[500],100,0.33
5,2026-04-22 00:00:00,18.15,18.15,18.15,57,3.31,3.72,0.00,0,804,Clouds,overcast clouds,[804],98,0.00
6,2026-04-22 03:00:00,16.56,16.56,16.56,58,1.28,0.90,0.00,0,804,Clouds,overcast clouds,[804],99,0.00
7,2026-04-22 06:00:00,17.43,17.43,17.43,56,0.92,0.55,0.00,0,804,Clouds,overcast clouds,[804],100,0.00


In [10]:
def weather_id_to_severity(weather_id):
    """
    Convert OWM weather_id to a base severity score (0-40).
    Combined with alert severity in the risk engine.
    Reference: https://openweathermap.org/weather-conditions
    """
    if 200 <= weather_id < 300:
        return 35
    elif 300 <= weather_id < 400:
        return 10
    elif 500 <= weather_id < 600:
        return {500: 15, 501: 20, 502: 30, 503: 35, 504: 40, 511: 25}.get(weather_id, 20)
    elif 600 <= weather_id < 700:
        return 25
    elif weather_id == 781:
        return 40
    elif 700 <= weather_id < 800:
        return 15
    return 0


# Apply severity score per timestamp first
df_forecast['severity_score'] = df_forecast['weather_id'].apply(weather_id_to_severity)

# Ensure date column exists (derived from timestamp)
df_forecast['date'] = df_forecast['timestamp'].dt.date

# Aggregate — worst day = highest severity score
daily_summary = df_forecast.groupby('date').agg(
    temp_min      =('temp_min_c',     'min'),
    temp_max      =('temp_max_c',     'max'),
    wind_max      =('wind_speed_ms',  'max'),
    rain_total    =('rain_3h_mm',     'sum'),
    snow_total    =('snow_3h_mm',     'sum'),
    max_pop       =('pop',            'max'),
    worst_severity=('severity_score', 'max'),
).reset_index()

# Recover the weather_id that produced the worst severity (for labelling)
worst_id_per_day = (
    df_forecast.sort_values('severity_score', ascending=False)
               .groupby('date', as_index=False)['weather_id']
               .first()
               .rename(columns={'weather_id': 'worst_weather_id'})
)

daily_summary = daily_summary.merge(worst_id_per_day, on='date')

print('Daily forecast summary:')
daily_summary

Daily forecast summary:


,date,temp_min,temp_max,wind_max,rain_total,snow_total,max_pop,worst_severity,worst_weather_id
0,2026-04-21,18.40,29.10,5.99,0.11,0,0.33,15,500
1,2026-04-22,16.56,28.33,8.35,0.00,0,0.00,0,804
2,2026-04-23,14.12,23.92,9.65,0.00,0,0.00,0,802
3,2026-04-24,13.23,24.32,4.26,0.46,0,0.38,15,500
4,2026-04-25,14.88,24.57,2.84,0.42,0,0.37,15,500
5,2026-04-26,15.58,17.30,1.92,0.00,0,0.00,0,803


---
## 5. Risk Score Engine — Preliminary

Combines OWM current conditions + One Call alerts into a 0-100 risk score.
This logic is implemented in `core/risk_engine.py`.

```
risk_score (0-100) = min(weather_severity + alert_severity + wind_bonus + rain_bonus, 100)
```

In [ ]:
def severity_to_score(severity_str):
    """Map severity label to numeric score (0-60)."""
    return {'Minor': 10, 'Moderate': 25, 'Severe': 45, 'Extreme': 60}.get(severity_str, 0)


def score_to_level(score):
    """Convert numeric score to risk level label."""
    if score >= 70: return 'CRITICAL'
    if score >= 45: return 'HIGH'
    if score >= 20: return 'ELEVATED'
    return 'LOW'


def compute_risk_score(weather_data, alert_list):
    """
    Combine OWM current data + One Call alerts into a 0-100 risk score.
    Preliminary version — production logic lives in core/risk_engine.py.
    """
    weather_severity = weather_id_to_severity(weather_data['weather_id'])

    alert_severity = max(
        (severity_to_score(a['severity']) for a in alert_list),
        default=0
    )

    wind = weather_data.get('wind_speed_ms', 0) or 0
    wind_bonus = 15 if wind >= 20 else 8 if wind >= 14 else 3 if wind >= 8 else 0

    rain = weather_data.get('rain_1h_mm', 0) or 0
    rain_bonus = 10 if rain >= 20 else 5 if rain >= 10 else 0

    total = min(weather_severity + alert_severity + wind_bonus + rain_bonus, 100)

    return {
        'risk_score': total,
        'risk_level': score_to_level(total),
        'breakdown': {
            'weather_severity': weather_severity,
            'alert_severity':   alert_severity,
            'wind_bonus':       wind_bonus,
            'rain_bonus':       rain_bonus,
        }
    }


risk_result = compute_risk_score(weather, alerts)

print('=' * 42)
print(f"  RISK SCORE : {risk_result['risk_score']}/100")
print(f"  RISK LEVEL : {risk_result['risk_level']}")
print('=' * 42)
print('\nBreakdown:')
for k, v in risk_result['breakdown'].items():
    print(f'  {k:<22} +{v}')

In [31]:
# Simulate scenarios to validate thresholds
print('Scenario simulation:\n')

scenarios = [
    {'name': 'Normal day',         'weather_id': 800, 'wind': 3,  'rain': 0,  'alert': 'Unknown'},
    {'name': 'Moderate rain',      'weather_id': 501, 'wind': 6,  'rain': 5,  'alert': 'Minor'},
    {'name': 'Storm + alert',      'weather_id': 502, 'wind': 18, 'rain': 15, 'alert': 'Moderate'},
    {'name': 'Severe emergency',   'weather_id': 212, 'wind': 25, 'rain': 25, 'alert': 'Severe'},
    {'name': 'Catastrophic event', 'weather_id': 781, 'wind': 35, 'rain': 40, 'alert': 'Extreme'},
]

for s in scenarios:
    fw = {'weather_id': s['weather_id'], 'wind_speed_ms': s['wind'], 'rain_1h_mm': s['rain']}
    fa = [{'severity': s['alert']}] if s['alert'] != 'Unknown' else []
    r  = compute_risk_score(fw, fa)
    print(f"  {s['name']:<26} score={r['risk_score']:>3}   level={r['risk_level']}")

Scenario simulation:

  Normal day                 score=  0   level=LOW
  Moderate rain              score= 30   level=ELEVATED
  Storm + alert              score= 68   level=HIGH
  Severe emergency           score=100   level=CRITICAL
  Catastrophic event         score=100   level=CRITICAL


---
## 6. Database Schema

In [ ]:
SCHEMA_SQL = """
CREATE TABLE IF NOT EXISTS weather_readings (
    id                  INTEGER PRIMARY KEY AUTOINCREMENT,
    timestamp           TEXT NOT NULL,
    city                TEXT,
    temp_c              REAL,
    feels_like_c        REAL,
    humidity_pct        INTEGER,
    pressure_hpa        INTEGER,
    visibility_m        INTEGER,
    wind_speed_ms       REAL,
    wind_gust_ms        REAL,
    rain_1h_mm          REAL DEFAULT 0,
    snow_1h_mm          REAL DEFAULT 0,
    weather_id          INTEGER,
    weather_main        TEXT,
    weather_desc        TEXT,
    clouds_pct          INTEGER,
    uvi                 REAL,
    created_at          TEXT DEFAULT (datetime('now'))
);

CREATE TABLE IF NOT EXISTS weather_alerts (
    id                  INTEGER PRIMARY KEY AUTOINCREMENT,
    source              TEXT DEFAULT 'aemet',
    title               TEXT,
    event               TEXT,
    sender_name         TEXT,
    severity            TEXT,
    urgency             TEXT,
    certainty           TEXT,
    onset               TEXT,
    expires             TEXT,
    description         TEXT,
    tags                TEXT,
    is_active           INTEGER DEFAULT 1,
    created_at          TEXT DEFAULT (datetime('now'))
);

CREATE TABLE IF NOT EXISTS risk_scores (
    id                  INTEGER PRIMARY KEY AUTOINCREMENT,
    timestamp           TEXT NOT NULL,
    score               INTEGER NOT NULL,
    level               TEXT NOT NULL,
    weather_component   INTEGER,
    alert_component     INTEGER,
    wind_bonus          INTEGER,
    rain_bonus          INTEGER,
    created_at          TEXT DEFAULT (datetime('now'))
);

CREATE TABLE IF NOT EXISTS kit_items (
    id                  INTEGER PRIMARY KEY AUTOINCREMENT,
    name                TEXT NOT NULL,
    category            TEXT,
    quantity            REAL NOT NULL,
    unit                TEXT,
    expiry_date         TEXT,
    eu_recommended      REAL,
    notes               TEXT,
    updated_at          TEXT DEFAULT (datetime('now'))
);
"""

conn = sqlite3.connect('haven_test.db')
conn.executescript(SCHEMA_SQL)
conn.commit()

tables = [r[0] for r in conn.execute("SELECT name FROM sqlite_master WHERE type='table'").fetchall()]
print(f'Database created: haven_test.db')
print(f'Tables: {tables}')

In [ ]:
# Insert current weather reading
conn.execute("""
    INSERT INTO weather_readings
    (timestamp, city, temp_c, feels_like_c, humidity_pct, pressure_hpa,
     visibility_m, wind_speed_ms, wind_gust_ms, rain_1h_mm, snow_1h_mm,
     weather_id, weather_main, weather_desc, clouds_pct, uvi)
    VALUES
    (:timestamp, :city, :temp_c, :feels_like_c, :humidity_pct, :pressure_hpa,
     :visibility_m, :wind_speed_ms, :wind_gust_ms, :rain_1h_mm, :snow_1h_mm,
     :weather_id, :weather_main, :weather_description, :clouds_pct, :uvi)
""", weather)

# Insert alerts
for a in alerts:
    conn.execute("""
        INSERT INTO weather_alerts
        (title, event, sender_name, severity, urgency, certainty, onset, expires, description, tags)
        VALUES (:title, :event, :sender_name, :severity, :urgency, :certainty,
                :onset, :expires, :description, :tags)
    """, {**a, 'tags': json.dumps(a['tags'])})

# Insert risk score
conn.execute("""
    INSERT INTO risk_scores
    (timestamp, score, level, weather_component, alert_component, wind_bonus, rain_bonus)
    VALUES (datetime('now'), :risk_score, :risk_level,
            :weather_severity, :alert_severity, :wind_bonus, :rain_bonus)
""", {
    'risk_score':       risk_result['risk_score'],
    'risk_level':       risk_result['risk_level'],
    **risk_result['breakdown']
})

conn.commit()
print('Data inserted into DB')

# Read back most recent records (ORDER BY rowid DESC avoids stale reads on re-runs)
row = conn.execute(
    'SELECT timestamp, temp_c, wind_speed_ms, weather_desc FROM weather_readings ORDER BY rowid DESC LIMIT 1'
).fetchone()
print(f'Weather row: {row}')

score_row = conn.execute(
    'SELECT score, level FROM risk_scores ORDER BY rowid DESC LIMIT 1'
).fetchone()
print(f'Risk score: {score_row[0]}/100 ({score_row[1]})')

---
## 7. Emergency Kit — Seed Data

Reference kit based on EU emergency preparedness recommendations (per person, 3 days).  
Source: https://ec.europa.eu/echo/what/civil-protection/eu-emergency-preparedness

In [ ]:
EU_KIT_REFERENCE = [
    {'name': 'Drinking water',          'category': 'water',     'eu_recommended': 9.0, 'unit': 'liters'},
    {'name': 'Water purification tabs', 'category': 'water',     'eu_recommended': 1.0, 'unit': 'units'},
    {'name': 'Non-perishable food',     'category': 'food',      'eu_recommended': 3.0, 'unit': 'days'},
    {'name': 'Can opener',              'category': 'food',      'eu_recommended': 1.0, 'unit': 'units'},
    {'name': 'First aid kit',           'category': 'meds',      'eu_recommended': 1.0, 'unit': 'units'},
    {'name': 'Regular medication',      'category': 'meds',      'eu_recommended': 7.0, 'unit': 'days'},
    {'name': 'Flashlight',              'category': 'tools',     'eu_recommended': 1.0, 'unit': 'units'},
    {'name': 'AA batteries',            'category': 'tools',     'eu_recommended': 6.0, 'unit': 'units'},
    {'name': 'Battery-powered radio',   'category': 'comms',     'eu_recommended': 1.0, 'unit': 'units'},
    {'name': 'Power bank',              'category': 'tools',     'eu_recommended': 1.0, 'unit': 'units'},
    {'name': 'Whistle',                 'category': 'tools',     'eu_recommended': 1.0, 'unit': 'units'},
    {'name': 'Sleeping bag',            'category': 'tools',     'eu_recommended': 1.0, 'unit': 'units'},
    {'name': 'Emergency blanket',       'category': 'tools',     'eu_recommended': 1.0, 'unit': 'units'},
    {'name': 'Document copies',         'category': 'documents', 'eu_recommended': 1.0, 'unit': 'units'},
    {'name': 'Cash',                    'category': 'documents', 'eu_recommended': 1.0, 'unit': 'units'},
    {'name': 'Basic hygiene kit',       'category': 'tools',     'eu_recommended': 1.0, 'unit': 'units'},
]

# Simulate a realistic kit with intentional gaps and expiry dates
current_kit = [{**item, 'quantity': item['eu_recommended'] * 0.6, 'expiry_date': None}
               for item in EU_KIT_REFERENCE]

current_kit[0]['quantity']    = 4.0           # Water: 4L vs 9L — gap!
current_kit[2]['expiry_date'] = '2026-04-20'  # Food expiring soon
current_kit[5]['expiry_date'] = '2027-01-01'  # Medication OK

# Clear previous runs to prevent row accumulation on re-execution
conn.execute("DELETE FROM kit_items")

for item in current_kit:
    conn.execute("""
        INSERT INTO kit_items (name, category, quantity, unit, expiry_date, eu_recommended)
        VALUES (:name, :category, :quantity, :unit, :expiry_date, :eu_recommended)
    """, item)
conn.commit()

print(f'{len(current_kit)} kit items inserted\n')

df_kit = pd.read_sql(
    'SELECT name, category, quantity, eu_recommended, unit, expiry_date FROM kit_items', conn
)
df_kit['gap']     = df_kit['eu_recommended'] - df_kit['quantity']
df_kit['gap_pct'] = (df_kit['gap'] / df_kit['eu_recommended'] * 100).round(1)
df_kit

In [16]:
today = datetime.today()

print('KIT GAPS (below EU recommendation):')
for _, row in df_kit[df_kit['gap'] > 0].iterrows():
    print(f"  {row['name']:<32} {row['quantity']:.1f} / {row['eu_recommended']:.1f} {row['unit']} ({row['gap_pct']}% missing)")

print('\nEXPIRING ITEMS (within 30 days):')
expiring = df_kit[df_kit['expiry_date'].notna()].copy()
expiring['days_to_expiry'] = expiring['expiry_date'].apply(
    lambda d: (datetime.strptime(d, '%Y-%m-%d') - today).days
)
critical = expiring[expiring['days_to_expiry'] <= 30]
for _, row in critical.iterrows():
    flag = 'CRITICAL' if row['days_to_expiry'] <= 7 else 'WARNING'
    print(f"  [{flag}] {row['name']:<32} expires in {row['days_to_expiry']} days ({row['expiry_date']})")
if critical.empty:
    print('  No items expiring within 30 days.')

KIT GAPS (below EU recommendation):
  Drinking water                   4.0 / 9.0 liters (55.6% missing)
  Water purification tabs          0.6 / 1.0 units (40.0% missing)
  Non-perishable food              1.8 / 3.0 days (40.0% missing)
  Can opener                       0.6 / 1.0 units (40.0% missing)
  First aid kit                    0.6 / 1.0 units (40.0% missing)
  Regular medication               4.2 / 7.0 days (40.0% missing)
  Flashlight                       0.6 / 1.0 units (40.0% missing)
  AA batteries                     3.6 / 6.0 units (40.0% missing)
  Battery-powered radio            0.6 / 1.0 units (40.0% missing)
  Power bank                       0.6 / 1.0 units (40.0% missing)
  Whistle                          0.6 / 1.0 units (40.0% missing)
  Sleeping bag                     0.6 / 1.0 units (40.0% missing)
  Emergency blanket                0.6 / 1.0 units (40.0% missing)
  Document copies                  0.6 / 1.0 units (40.0% missing)
  Cash                     

---
## 8. Summary

In [ ]:
print('=' * 55)
print('  HAVEN — DATA RETRIEVAL VALIDATION SUMMARY')
print('=' * 55)

print('\n[OK] One Call API 3.0 — current conditions')
print(f"     Temp: {weather['temp_c']}C | Wind: {weather['wind_speed_ms']}m/s | {weather['weather_description']}")

print('\n[OK] One Call API 3.0 — national alerts (AEMET)')
print(f'     {len(alerts)} active alert(s)')

print('\n[OK] Forecast API — 5 days / 3h')
print(f'     {len(df_forecast)} timestamps | {len(daily_summary)} days aggregated')

print('\n[OK] Risk Score Engine (preliminary)')
print(f"     Score: {risk_result['risk_score']}/100 ({risk_result['risk_level']})")

print('\n[OK] SQLite DB — schema validated')
print(f'     Tables: weather_readings, weather_alerts, risk_scores, kit_items')

print(f'\n[OK] Emergency kit — {len(current_kit)} items')
gaps_count = len(df_kit[df_kit['gap'] > 0])
print(f'     {gaps_count} gap(s) | {len(critical)} item(s) expiring within 30 days')

print('\n' + '=' * 55)
print('  NEXT STEPS')
print('=' * 55)
print("""
  1. Move fetchers to data/fetchers/ as Python modules
  2. Create data/db.py with DB access functions
  3. Build core/risk_engine.py with logic validated here
  4. Build core/inventory_analyzer.py (gap + expiry detection)
  5. Build core/alert_prioritizer.py (risk x gaps -> ranked actions)
""")

conn.close()
print('DB connection closed.')

---
## 9. Regional Risk — GDACS + ReliefWeb

Two independent free sources replace the ACLED API (which requires a paid license):

- **GDACS** (UN + European Commission) — natural disasters near the region
- **ReliefWeb** (UN OCHA) — conflict and humanitarian crisis reports

Score: 0-30 | Independent signal — never summed with weather or health.

In [18]:
import sys
for key in list(sys.modules.keys()):
    if 'regional' in key:
        del sys.modules[key]

from core.regional_risk_fetcher import (
    get_regional_snapshot,
    simulate_regional_snapshot,
    regional_score_to_level,
)

print('Regional risk fetcher ready')

Regional risk fetcher ready


In [19]:
import sys
for key in list(sys.modules.keys()):
    if 'regional' in key:
        del sys.modules[key]

from core.regional_risk_fetcher import get_regional_snapshot, simulate_regional_snapshot, regional_score_to_level

In [ ]:
try:
    geo_snapshot = get_regional_snapshot(country=COUNTRY)
    print('Live data fetched')
except Exception as e:
    print(f'Fetch failed ({e}) — using simulated snapshot')
    geo_snapshot = simulate_regional_snapshot('calm')

print(f'\n{"="*48}')
print(f'  REGIONAL RISK SIGNAL')
print(f'  Score          : {geo_snapshot.regional_score}/30')
print(f'  Level          : {geo_snapshot.level}')
print(f'  Disaster score : {geo_snapshot.disaster_score}/15 ({len(geo_snapshot.disaster_events)} events)')
print(f'  Crisis score   : {geo_snapshot.crisis_score}/15 ({len(geo_snapshot.crisis_reports)} reports)')
print(f'{"="*48}')

if geo_snapshot.disaster_events:
    print('\nActive disasters:')
    for e in geo_snapshot.disaster_events[:3]:
        print(f'  [{e.alert_level:<6}] {e.event_type} — {e.country}: {e.title[:60]}')

if geo_snapshot.crisis_reports:
    print('\nCrisis reports:')
    for r in geo_snapshot.crisis_reports[:3]:
        print(f'  [{r.country:<12}] {r.theme}: {r.title[:55]}')

# Expose flat variables for HavenSignals and prioritizer
geo_score   = geo_snapshot.regional_score
geo_trend   = "STABLE"
geo_country = geo_snapshot.country

In [ ]:
conn = sqlite3.connect('haven_test.db')
conn.execute("""
    CREATE TABLE IF NOT EXISTS regional_snapshots (
        id               INTEGER PRIMARY KEY AUTOINCREMENT,
        country          TEXT,
        disaster_score   INTEGER,
        crisis_score     INTEGER,
        regional_score   INTEGER,
        level            TEXT,
        disaster_events  INTEGER,
        crisis_reports   INTEGER,
        fetched_at       TEXT,
        created_at       TEXT DEFAULT (datetime('now'))
    )
""")
conn.execute("""
    INSERT INTO regional_snapshots
    (country, disaster_score, crisis_score, regional_score, level,
     disaster_events, crisis_reports, fetched_at)
    VALUES (?,?,?,?,?,?,?,?)
""", (
    geo_snapshot.country,
    geo_snapshot.disaster_score,
    geo_snapshot.crisis_score,
    geo_snapshot.regional_score,
    geo_snapshot.level,
    len(geo_snapshot.disaster_events),
    len(geo_snapshot.crisis_reports),
    geo_snapshot.fetched_at,
))
conn.commit()
conn.close()
print('Regional snapshot stored in DB')

In [ ]:
# Update the geo fields in HavenSignals to use the new regional snapshot
# Run this BEFORE the Section 11 display cell

geo_score   = geo_snapshot.regional_score
geo_trend   = "STABLE"   # RegionalSnapshot doesn't compute trend — always STABLE for now
geo_country = geo_snapshot.country

print(f'Geo score ready: {geo_score}/30 ({geo_snapshot.level})')

In [23]:
import pytest
pytest.main(['tests/test_regional_risk_fetcher.py', '-v', '--tb=short', '--no-header'])

============================= test session starts ==============================
collecting ... collected 39 items

tests/test_regional_risk_fetcher.py::TestRegionalScoreToLevel::test_minimal PASSED [  2%]
tests/test_regional_risk_fetcher.py::TestRegionalScoreToLevel::test_low PASSED [  5%]
tests/test_regional_risk_fetcher.py::TestRegionalScoreToLevel::test_medium PASSED [  7%]
tests/test_regional_risk_fetcher.py::TestRegionalScoreToLevel::test_high PASSED [ 10%]
tests/test_regional_risk_fetcher.py::TestRegionalScoreToLevel::test_boundary_values PASSED [ 12%]
tests/test_regional_risk_fetcher.py::TestComputeDisasterScore::test_no_events_returns_zero PASSED [ 15%]
tests/test_regional_risk_fetcher.py::TestComputeDisasterScore::test_single_green_event PASSED [ 17%]
tests/test_regional_risk_fetcher.py::TestComputeDisasterScore::test_single_red_event PASSED [ 20%]
tests/test_regional_risk_fetcher.py::TestComputeDisasterScore::test_score_capped_at_15 PASSED [ 23%]
tests/test_regional_risk_fet

<ExitCode.OK: 0>

---
## 10. Health Risk — ECDC CDTR Integration (Option C)

Fetches the latest ECDC Communicable Disease Threats Report (CDTR),
extracts active health threats, and computes an independent health
risk score (0-50).

Source: https://www.ecdc.europa.eu/en/publications-and-data/monitoring/weekly-threats-reports
Published: every Thursday. No authentication required.

Option C design — three independent signals, never summed:
    🌤 Weather Risk    0-100  (fast-moving, OWM One Call)
    ⚔  Regional Risk   0-30   (weekly, GDACS + ReliefWeb)
    🦠 Health Risk     0-50   (weekly, ECDC CDTR)

In [24]:
from core.health_fetcher import (
    fetch_latest_cdtr_summary,
    _extract_threats_from_text,
    _compute_health_score,
    get_health_snapshot,
    simulate_health_snapshot
)

# Fetch live from ECDC (no credentials needed)
try:
    health_snapshot = get_health_snapshot(timeout=15)
    print(f'Live CDTR fetched: {health_snapshot.week_label}')
except Exception as e:
    print(f'Live fetch failed ({e}) — using simulated snapshot')
    health_snapshot = simulate_health_snapshot('routine')

print(f'\n{"="*50}')
print(f'  HEALTH RISK')
print(f'  Period      : {health_snapshot.period}')
print(f'  Score       : {health_snapshot.health_score}/50')
print(f'  Level       : {health_snapshot.level}')
print(f'  Top threats : {", ".join(health_snapshot.top_threats[:3]) or "None"}')
print(f'  Source      : {health_snapshot.cdtr_url}')
print(f'{"="*50}')

Live CDTR fetched: Week 16, 2026

  HEALTH RISK
  Period      : 12-18 April 2026
  Score       : 16/50
  Level       : MEDIUM
  Top threats : Measles, Chikungunya, Respiratory
  Source      : https://www.ecdc.europa.eu/en/publications-data/communicable-disease-threats-report-12-18-april-2026-week-16


In [ ]:
import json as json_lib

conn = sqlite3.connect('haven_test.db')
conn.execute("""
    CREATE TABLE IF NOT EXISTS health_snapshots (
        id              INTEGER PRIMARY KEY AUTOINCREMENT,
        week_label      TEXT,
        period          TEXT,
        health_score    INTEGER,
        level           TEXT,
        top_threats     TEXT,   -- JSON array
        cdtr_url        TEXT,
        fetched_at      TEXT,
        created_at      TEXT DEFAULT (datetime('now'))
    )
""")
conn.execute("""
    INSERT INTO health_snapshots
    (week_label, period, health_score, level, top_threats, cdtr_url, fetched_at)
    VALUES (?,?,?,?,?,?,?)
""", (
    health_snapshot.week_label,
    health_snapshot.period,
    health_snapshot.health_score,
    health_snapshot.level,
    json_lib.dumps(health_snapshot.top_threats),
    health_snapshot.cdtr_url,
    health_snapshot.fetched_at,
))
conn.commit()
print('Health snapshot stored in DB')
conn.close()

In [26]:
pytest.main(['tests/test_health_fetcher.py', '-v', '--tb=short', '--no-header'])

============================= test session starts ==============================
collecting ... collected 34 items

tests/test_health_fetcher.py::TestHealthScoreToLevel::test_routine PASSED [  2%]
tests/test_health_fetcher.py::TestHealthScoreToLevel::test_medium PASSED [  5%]
tests/test_health_fetcher.py::TestHealthScoreToLevel::test_high PASSED   [  8%]
tests/test_health_fetcher.py::TestHealthScoreToLevel::test_critical PASSED [ 11%]
tests/test_health_fetcher.py::TestHealthScoreToLevel::test_boundary_values PASSED [ 14%]
tests/test_health_fetcher.py::TestComputeHealthScore::test_no_threats_returns_zero PASSED [ 17%]
tests/test_health_fetcher.py::TestComputeHealthScore::test_single_threat PASSED [ 20%]
tests/test_health_fetcher.py::TestComputeHealthScore::test_score_capped_at_max PASSED [ 23%]
tests/test_health_fetcher.py::TestComputeHealthScore::test_diminishing_returns PASSED [ 26%]
tests/test_health_fetcher.py::TestComputeHealthScore::test_dominant_threat_outweighs_many_small PASSED

<ExitCode.OK: 0>

---
## 11. Three signals display



In [ ]:
from core.risk_engine import RiskResult, HavenSignals, _geo_level

# Rebuild RiskResult from the existing dict
risk_result_obj = RiskResult(
    risk_score=       risk_result['risk_score'],
    risk_level=       risk_result['risk_level'],
    weather_severity= risk_result['breakdown']['weather_severity'],
    alert_severity=   risk_result['breakdown']['alert_severity'],
    wind_bonus=       risk_result['breakdown']['wind_bonus'],
    rain_bonus=       risk_result['breakdown']['rain_bonus'],
)

signals = HavenSignals(
    weather=            risk_result_obj,
    geo_score=          geo_score,          
    geo_trend=          geo_trend,
    geo_country=        geo_country,
    health_score=       health_snapshot.health_score,
    health_level=       health_snapshot.level,
    top_health_threats= health_snapshot.top_threats,
)

summary = signals.summary()

print('HAVEN — THREE INDEPENDENT RISK SIGNALS\n')
print(f'  🌤  Weather        {summary["weather"]["score"]:>4}/100   {summary["weather"]["level"]}')
print(f'  ⚔  Regional Risk  {summary["geopolitical"]["score"]:>4}/30    {summary["geopolitical"]["level"]}  (trend: {summary["geopolitical"]["trend"]})')
print(f'  🦠 Health         {summary["health"]["score"]:>4}/50    {summary["health"]["level"]}')
if summary["health"]["top_threats"]:
    print(f'       Active threats: {", ".join(summary["health"]["top_threats"][:3])}')
print()
print('  ℹ  Signals are independent — not summed.')

---
## 12. Alert prioritizer

Crosses all three risk signals × kit inventory → ranked action list.

In [28]:
# 3. Prioritized action list — all three signals × kit
import sys
for key in list(sys.modules.keys()):
    if 'alert_prioritizer' in key:
        del sys.modules[key]

from core.alert_prioritizer import prioritize
# Rebuild inv_report from the kit seed data (Section 7)
from core.inventory_analyzer import KitItem, analyze_inventory
from datetime import date as _date

kit_objects = []
for item in current_kit:
    exp = item.get('expiry_date')
    kit_objects.append(KitItem(
        name=           item['name'],
        category=       item['category'],
        quantity=       item['quantity'],
        unit=           item['unit'],
        eu_recommended= item['eu_recommended'],
        expiry_date=    _date.fromisoformat(exp) if exp else None,
    ))
    
inv_report = analyze_inventory(kit_objects)

action_list = prioritize(
    risk=               risk_result_obj,
    inventory_report=   inv_report,
    geo_score=          geo_score,
    geo_trend=          geo_trend,
    geo_country=        geo_country,
    health_score=       health_snapshot.health_score,
    health_level=       health_snapshot.level,
    top_health_threats= health_snapshot.top_threats,
)

print(f'Total actions: {len(action_list)}\n')
for i, a in enumerate(action_list, 1):
    print(f'{i:>2}. [{a.urgency:<10}] [{a.category:<12}] score={a.priority_score}')
    print(f'    {a.message}')
    print()

Total actions: 27

 1. [IMMEDIATE ] [EXPIRY      ] score=80
    Replace Non-perishable food: expires in -1 day(s) (2026-04-20).

 2. [SOON      ] [KIT_GAP     ] score=55
    Replenish Drinking water: 4.0/9.0 liters (56% missing).

 3. [SOON      ] [COMBINED    ] score=40
    Weather ELEVATED + low Drinking water stock (4.0/9.0 liters): restock before conditions worsen.

 4. [SOON      ] [KIT_GAP     ] score=40
    Replenish Water purification tabs: 0.6/1.0 units (40% missing).

 5. [SOON      ] [KIT_GAP     ] score=40
    Replenish Non-perishable food: 1.8/3.0 days (40% missing).

 6. [SOON      ] [KIT_GAP     ] score=40
    Replenish Can opener: 0.6/1.0 units (40% missing).

 7. [SOON      ] [KIT_GAP     ] score=40
    Replenish First aid kit: 0.6/1.0 units (40% missing).

 8. [SOON      ] [KIT_GAP     ] score=40
    Replenish Regular medication: 4.2/7.0 days (40% missing).

 9. [ROUTINE   ] [HEALTH_KIT  ] score=39
    Measles, Chikungunya active + low First aid kit stock (0.6/1.0 uni

## 13. Tests

In [29]:
import pytest

exit_code = pytest.main([
    'tests/',
    '-v',
    '--tb=short',
    '--no-header',
])
print(f'\npytest exit code: {exit_code} (0 = all passed)')

============================= test session starts ==============================
collecting ... collected 145 items / 1 error

==================================== ERRORS ====================================
_________ ERROR collecting tests/test_geopolitical_fetcher_archived.py _________
ImportError while importing test module '/home/ildebrando/code/ijesusjr/000_DS_Portfolio/02_prepsense/tests/test_geopolitical_fetcher_archived.py'.
Hint: make sure your test modules/packages have valid Python names.
Traceback:
../../../../.pyenv/versions/3.12.9/lib/python3.12/importlib/__init__.py:90: in import_module
    return _bootstrap._gcd_import(name[level:], package, level)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
tests/test_geopolitical_fetcher_archived.py:18: in <module>
    from core.geopolitical_fetcher import (
E   ModuleNotFoundError: No module named 'core.geopolitical_fetcher'
=========================== short test summary info ============================
ERROR tes